In [1]:

import io
import json
import os
import subprocess
import tempfile
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
 
os.environ.setdefault("OMP_THREAD_LIMIT", "1")  
 
import fitz  # pymupdf
import pytesseract
from docx import Document
from PIL import Image
from tqdm.auto import tqdm
 
OCR_LANG = "rus+eng"
OCR_DPI = 300
MIN_PAGE_CHARS = 50     
MIN_DOCX_CHARS = 200    
 
RAW_DIR = Path("./mipt_tz_dataset/raw")
OUT_DIR = Path("./mipt_tz_dataset/processed")
INDEX_FILE = Path("./mipt_tz_dataset/documents_index.jsonl")

/home/ems/ml311/lib/python3.11/site-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [2]:
pytesseract.get_tesseract_version()  # упадёт сразу, если tesseract не найден
assert "rus" in pytesseract.get_languages(), "не установлен языковой пакет rus"

In [3]:

def ocr_image(img: Image.Image) -> str:
    return pytesseract.image_to_string(img.convert("RGB"), lang=OCR_LANG, config="--psm 3")
 
 
def ocr_pdf_page(page) -> str:
    pix = page.get_pixmap(dpi=OCR_DPI)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    return ocr_image(img)
 
 
def extract_pdf(path):
    parts = []
    with fitz.open(path) as doc:
        for page_num, page in enumerate(doc, start=1):
            text = page.get_text()
            tag = ""
            if len(text.strip()) < MIN_PAGE_CHARS:
                try:
                    text = ocr_pdf_page(page)
                    tag = " (OCR)"
                except Exception as e:
                    print("OCR ERROR:", path, page_num, e)
                    text = ""
            if text.strip():
                parts.append(f"\n\n=== PAGE {page_num}{tag} ===\n\n" + text)
    return "".join(parts)

In [4]:

def extract_docx(path):
    doc = Document(path)
    result = [p.text for p in doc.paragraphs if p.text.strip()]
 
    for table in doc.tables:
        for row in table.rows:
            cells = []
            for c in row.cells:
                t = c.text.strip()
                if t and (not cells or cells[-1] != t): 
                    cells.append(t)
            if cells:
                result.append(" | ".join(cells))
 
    text = "\n".join(result)
 
    if len(text.strip()) < MIN_DOCX_CHARS:
        ocr_parts = []
        for rel in doc.part.rels.values():
            if "image" not in rel.reltype:
                continue
            try:
                img = Image.open(io.BytesIO(rel.target_part.blob))
                if min(img.size) < 200:  
                    continue
                ocr_parts.append(ocr_image(img))
            except Exception:
                continue 
        ocr_text = "\n\n".join(t for t in ocr_parts if t.strip())
        if len(ocr_text.strip()) > len(text.strip()):
            text = (text + "\n\n=== OCR ===\n\n" + ocr_text).strip()
 
    return text
 

In [5]:

def extract_doc(path):
    try:
        with tempfile.TemporaryDirectory() as tmp:
            subprocess.run(
                [
                    "libreoffice",
                    f"-env:UserInstallation=file://{tmp}/profile",  # свой профиль -> можно параллелить
                    "--headless", "--convert-to", "docx",
                    "--outdir", tmp, str(path),
                ],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=180,
            )
            converted = Path(tmp) / (path.stem + ".docx")
            if converted.exists():
                return extract_docx(converted)
    except Exception as e:
        print("DOC ERROR:", path, e)
    return ""
 
 
def extract_text(path):
    ext = path.suffix.lower()
    if ext == ".pdf":
        return extract_pdf(path)
    if ext == ".docx":
        return extract_docx(path)
    if ext == ".doc":
        return extract_doc(path)
    return ""

In [6]:

def process_document(meta):
    src = Path(meta["path"])
    if not src.exists():
        return None
 
    text = extract_text(src)
    if len(text.strip()) < 100:
        return None
 
    purchase_dir = OUT_DIR / meta["purchase"]
    purchase_dir.mkdir(parents=True, exist_ok=True)
 
    txt_file = purchase_dir / f"{src.stem}.txt"
    txt_file.write_text(text, encoding="utf-8")
 
    result = {**meta, "processed_file": str(txt_file), "text_length": len(text)}
    (purchase_dir / f"{src.stem}.json").write_text(
        json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return result
 
 
def safe_process(meta):
    try:
        return process_document(meta)
    except Exception as e:
        return {"_error": str(e), "path": meta.get("path")}

In [7]:

if __name__ == "__main__":
    documents = []
    with open(INDEX_FILE, encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            if item.get("status") == "downloaded":
                documents.append(item)
 
    def already_done(meta):
        src = Path(meta["path"])
        return (OUT_DIR / meta["purchase"] / f"{src.stem}.json").exists()
 
    todo = [d for d in documents if not already_done(d)]
    print(f"Всего: {len(documents)}, уже обработано: {len(documents) - len(todo)}, к OCR/обработке: {len(todo)}")
 
    errors, new_ok = [], 0
    with ProcessPoolExecutor(max_workers=10) as pool:  # 9900X: 12 ядер, 2 оставляем системе
        for res in tqdm(pool.map(safe_process, todo, chunksize=1), total=len(todo)):
            if res is None:
                continue
            if "_error" in res:
                errors.append(res)
            else:
                new_ok += 1
 
    print("Новых обработано:", new_ok, "| ошибок:", len(errors))
 
    all_items = []
    for jf in OUT_DIR.glob("*/*.json"):
        all_items.append(json.loads(jf.read_text(encoding="utf-8")))
    with open(OUT_DIR / "processed_index.jsonl", "w", encoding="utf-8") as f:
        for item in all_items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print("В индексе:", len(all_items))

Всего: 5312, уже обработано: 0, к OCR/обработке: 5312


  0%|          | 0/5312 [00:00<?, ?it/s]

DOC ERROR: mipt_tz_dataset/raw/31401636806/01_ДОКУМЕНТАЦИЯ_ЗТ14-070_эл.в..doc Command '['libreoffice', '-env:UserInstallation=file:///tmp/tmpm7jg5_uk/profile', '--headless', '--convert-to', 'docx', '--outdir', '/tmp/tmpm7jg5_uk', 'mipt_tz_dataset/raw/31401636806/01_ДОКУМЕНТАЦИЯ_ЗТ14-070_эл.в..doc']' timed out after 179.9999916740053 seconds
DOC ERROR: mipt_tz_dataset/raw/31401636806/02_ДОКУМЕНТАЦИЯ_ЗТ14-070_эл.в.[1].doc Command '['libreoffice', '-env:UserInstallation=file:///tmp/tmpdi5doa9l/profile', '--headless', '--convert-to', 'docx', '--outdir', '/tmp/tmpdi5doa9l', 'mipt_tz_dataset/raw/31401636806/02_ДОКУМЕНТАЦИЯ_ЗТ14-070_эл.в.[1].doc']' timed out after 179.99999068200123 seconds
Новых обработано: 5277 | ошибок: 0
В индексе: 5277
